In [1]:
import numpy as np
import pandas as pd 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [2]:
from gensim.models import Word2Vec

In [3]:
import torch
import math

# **self-attention class**

In [4]:
class SelfAttention:

    # to initialize it
    def __init__(self, dimen=4, heads=1, random_w=True):

        self.dimen = dimen
        self.random_w = random_w
        self.heads = heads

        if random_w:
            self.w_query = torch.rand(dimen, dimen//heads).float()
            self.w_key = torch.rand(dimen, dimen//heads).float()
            self.w_value = torch.rand(dimen, dimen//heads).float()
        else:
            self.w_query = None
            self.w_key = None
            self.w_value = None

    def getQKV(self, static_embeddings):

        if self.random_w == False:
            
            queries = static_embeddings.float()
            keys = static_embeddings.float()
            values = static_embeddings.float()
            
        else:
        
            queries = torch.matmul(static_embeddings.float(), self.w_query)
            keys = torch.matmul(static_embeddings.float(), self.w_key)
            values = torch.matmul(static_embeddings.float(), self.w_value)

        return queries, keys, values

    def dotproduct(self, queries, keys):

        dot_product = torch.matmul(queries,keys.T)
        return dot_product

    def get_sqrt(self, dot_product):

        sqroot = math.sqrt(self.dimen)
        return (1/sqroot)*dot_product

    def mask_weights(self, tensor):
        tensor = tensor.float().clone()
        mask = torch.triu(torch.ones_like(tensor), diagonal=1).bool()
        tensor[mask] = float('-inf')
        return tensor

    def apply_softmax(self, scaled_dp):
        
        weights = torch.softmax(scaled_dp.T, dim=0).T
        return weights

    def mul_weights_and_values(self, weights, values):

        contextual_embeddings = torch.matmul(weights, values)
        #print(contextual_embeddings)
        return contextual_embeddings
        

    # get embeddings of different sentences
    def __call__(self, static_embeddings):

        # 1. obtain query, key, value vectors
        queries, keys, values = self.getQKV(static_embeddings)

        # 2. perform dot product bw query and key
        dot_product = self.dotproduct(queries,keys)

        # 3. scale by 1/sqrt(dimen)
        scaled_dp = self.get_sqrt(dot_product)

        # 4. mask the weights
        masked = self.mask_weights(scaled_dp)

        # 5. apply softmax to each row
        weights = self.apply_softmax(masked)

        # 6. multiply w value to get final embeddings
        contextual_embeddings = self.mul_weights_and_values(weights,values)
        return contextual_embeddings

In [5]:
sentence1 = "River bank flows"
s1 = sentence1.split(" ")
print(s1)

['River', 'bank', 'flows']


In [6]:
sentence2 = "Money bank grows"
s2 = sentence2.split(" ")
print(s2)

['Money', 'bank', 'grows']


In [7]:
sentences = [s1, s2]
print(sentences)

[['River', 'bank', 'flows'], ['Money', 'bank', 'grows']]


In [8]:
model = Word2Vec(sentences, vector_size=4, window=5, min_count=1, sg=1)

In [9]:
input_matrix1 = []

for word in s1:
    embedding = model.wv[word]
    input_matrix1.append(embedding)

np_ip1 = np.array(input_matrix1)
static_embeddings1 = torch.tensor(np_ip1)

#print(query_tensors)

In [10]:
sa_block = SelfAttention(4,True)

In [11]:
ce1 = sa_block(static_embeddings1)

In [12]:
input_matrix2 = []

for word in s2:
    embedding = model.wv[word]
    input_matrix2.append(embedding)

np_ip2 = np.array(input_matrix2)
static_embeddings2 = torch.tensor(np_ip2)

In [13]:
ce2 = sa_block(static_embeddings2)

In [14]:
print(static_embeddings2[0:2])

tensor([[-0.1254, -0.0941,  0.1845, -0.0383],
        [-0.0134,  0.0059,  0.1276,  0.2252]])


# **multi-head attention**

In [15]:
ambiguous_sentence = "The cat chased the mouse until it stumbled"
ambiguous_sentence = ambiguous_sentence.split(" ")
print(ambiguous_sentence)

['The', 'cat', 'chased', 'the', 'mouse', 'until', 'it', 'stumbled']


In [16]:
def get_embeddings(model, sentence):
    
    embeddings = [model.wv[word] for word in sentence]
    embeddings_np = np.array(embeddings)
    return torch.tensor(embeddings_np)

In [17]:
model = Word2Vec([ambiguous_sentence], vector_size=512, window=5, min_count=1, sg=1)

In [18]:
emb = get_embeddings(model,ambiguous_sentence)
print(emb.shape)

torch.Size([8, 512])


In [19]:
print(emb)

tensor([[-8.3858e-04, -1.2931e-03, -2.7027e-04,  ..., -1.9349e-03,
         -2.9324e-04,  1.3476e-03],
        [ 1.2903e-03, -2.2377e-04,  1.4973e-03,  ...,  1.0317e-04,
         -1.7566e-03,  1.6347e-03],
        [ 1.1490e-03, -5.7839e-04,  6.1753e-04,  ..., -1.6250e-03,
         -2.8566e-05, -5.1714e-04],
        ...,
        [-1.2890e-03,  8.4774e-04, -9.2673e-05,  ...,  1.8470e-03,
         -1.1359e-03,  1.6143e-03],
        [-1.4161e-03, -1.8756e-03, -5.3587e-04,  ..., -5.0609e-04,
          1.4158e-03, -6.7645e-04],
        [-1.0473e-04,  4.6178e-05,  9.9675e-04,  ..., -1.3424e-03,
         -9.7645e-04, -4.4665e-04]])


In [20]:
sa1 = SelfAttention(512,8,True)
sa2 = SelfAttention(512,8,True)
sa3 = SelfAttention(512,8,True)
sa4 = SelfAttention(512,8,True)
sa5 = SelfAttention(512,8,True)
sa6 = SelfAttention(512,8,True)
sa7 = SelfAttention(512,8,True)
sa8 = SelfAttention(512,8,True)

In [21]:
e1 = sa1(emb)
e2 = sa2(emb)
e3 = sa3(emb)
e4 = sa4(emb)
e5 = sa5(emb)
e6 = sa6(emb)
e7 = sa7(emb)
e8 = sa8(emb)

In [22]:
print(e1.shape)

torch.Size([8, 64])


In [23]:
print(e2)

tensor([[ 1.7734e-02, -2.7084e-03,  5.9512e-03, -1.6586e-02, -3.2360e-03,
         -3.8054e-03, -6.1311e-03, -1.5374e-03, -2.5335e-03, -9.2204e-03,
         -4.5414e-03, -7.5729e-03, -1.1698e-02, -6.3652e-04, -6.5695e-03,
          7.2289e-05, -1.3140e-03, -8.3116e-03,  4.1890e-04,  6.4392e-03,
          8.3850e-04, -1.1952e-02, -7.9490e-03, -1.0976e-02, -1.3890e-02,
         -6.0548e-03, -5.0777e-04, -5.9332e-03, -1.1600e-03, -3.8913e-03,
          2.5319e-04, -9.5026e-03,  9.0270e-03, -2.7812e-03,  1.4488e-02,
          3.6928e-03, -2.8937e-03, -2.4484e-03, -1.6391e-03, -1.0642e-02,
         -2.4041e-03, -6.5777e-03,  5.2898e-03,  4.7390e-03, -7.2257e-03,
         -2.5461e-03,  4.8474e-03, -6.1340e-03, -1.0120e-02, -1.4955e-02,
         -8.0720e-03, -2.4590e-03,  3.8255e-03, -1.3756e-02, -1.1400e-02,
         -7.1091e-03, -1.0423e-02, -2.3766e-03,  6.2427e-03, -1.4609e-03,
         -1.0816e-02, -4.2241e-03,  2.6481e-03,  1.6414e-03],
        [ 1.1493e-02,  7.8107e-03,  7.5195e-03, -2

In [24]:
concatenated = torch.cat([e1,e2,e3,e4,e5,e6,e7,e8], dim=1)
print(concatenated.shape)

torch.Size([8, 512])


In [25]:
print(concatenated)

tensor([[ 0.0038, -0.0066, -0.0171,  ..., -0.0064, -0.0016, -0.0132],
        [ 0.0053, -0.0056, -0.0043,  ..., -0.0019,  0.0029, -0.0034],
        [-0.0044, -0.0054, -0.0110,  ..., -0.0076, -0.0016, -0.0088],
        ...,
        [ 0.0049,  0.0021,  0.0037,  ...,  0.0039,  0.0075,  0.0046],
        [ 0.0034, -0.0004,  0.0028,  ...,  0.0028,  0.0053,  0.0019],
        [ 0.0035,  0.0012,  0.0035,  ...,  0.0033,  0.0070,  0.0030]])


In [26]:
class MultiHeadAttention:
    
    def __init__(self, num_heads=1, dim=4, random_w=True):
        
        self.num_heads = num_heads
        self.dim = dim
        self.random_w = random_w

    def __call__(self, emb):
        
        outputs = []
        
        for i in range(self.num_heads):
            sa = SelfAttention(self.dim, self.num_heads, self.random_w)
            ce = sa(emb)
            outputs.append(ce)

        concatenated = torch.cat(outputs, dim=1)
        return concatenated

In [27]:
ma = MultiHeadAttention(8,512,True)
embb = ma(emb)
print(embb.shape)

torch.Size([8, 512])


In [28]:
print(embb)

tensor([[ 0.0169, -0.0053,  0.0010,  ..., -0.0061,  0.0088, -0.0111],
        [ 0.0165, -0.0003,  0.0092,  ...,  0.0024,  0.0052, -0.0046],
        [ 0.0036, -0.0068, -0.0003,  ..., -0.0016, -0.0002, -0.0069],
        ...,
        [ 0.0125,  0.0047,  0.0081,  ...,  0.0113,  0.0118,  0.0021],
        [ 0.0102,  0.0031,  0.0065,  ...,  0.0066,  0.0086,  0.0006],
        [ 0.0109,  0.0039,  0.0081,  ...,  0.0079,  0.0083,  0.0021]])


# **positional encoding**

In [29]:
print(math.sin(1))
print(math.cos(1))

0.8414709848078965
0.5403023058681398


In [30]:
for i in range(2):
    print(i)

0
1


In [31]:
pos = 1
dim = 6

for word in emb:
    pos_enc = []
    for i in range(dim//2): # 1 pair
        pos_enc.append(math.sin(pos/pow(10000,(2*i/dim))))
        pos_enc.append(math.cos(pos/pow(10000,(2*i/dim))))
    print(pos_enc)

[0.8414709848078965, 0.5403023058681398, 0.046399223464731285, 0.9989229760406304, 0.0021544330233656045, 0.9999976792064809]
[0.8414709848078965, 0.5403023058681398, 0.046399223464731285, 0.9989229760406304, 0.0021544330233656045, 0.9999976792064809]
[0.8414709848078965, 0.5403023058681398, 0.046399223464731285, 0.9989229760406304, 0.0021544330233656045, 0.9999976792064809]
[0.8414709848078965, 0.5403023058681398, 0.046399223464731285, 0.9989229760406304, 0.0021544330233656045, 0.9999976792064809]
[0.8414709848078965, 0.5403023058681398, 0.046399223464731285, 0.9989229760406304, 0.0021544330233656045, 0.9999976792064809]
[0.8414709848078965, 0.5403023058681398, 0.046399223464731285, 0.9989229760406304, 0.0021544330233656045, 0.9999976792064809]
[0.8414709848078965, 0.5403023058681398, 0.046399223464731285, 0.9989229760406304, 0.0021544330233656045, 0.9999976792064809]
[0.8414709848078965, 0.5403023058681398, 0.046399223464731285, 0.9989229760406304, 0.0021544330233656045, 0.9999976792

In [32]:
def get_emb(model, word, pos, dim):
        
    emb = model.wv[word]
    
    pos_enc = np.zeros(dim)
    
    for i in range(dim//2): # 1 pair
        pos_enc[2*i] = math.sin(pos/pow(10000,(2*i/dim)))
        pos_enc[2*i+1] = math.cos(pos/pow(10000,(2*i/dim)))

    emb_np = np.array(emb)
    emb_t = torch.tensor(emb_np)

    pos_np = np.array(pos_enc)
    pos_t = torch.tensor(pos_np)

    result = torch.add(emb_t, pos_t)
    return result


def generate_pos_encodings(model,sentence):

    embeddings = [get_emb(model,word,pos,512) for pos,word in enumerate(sentence)]
    return torch.stack(embeddings)

In [33]:
enc = generate_pos_encodings(model,ambiguous_sentence)
print(enc.shape)

torch.Size([8, 512])


In [34]:
apply_ma = MultiHeadAttention(8,512,True)
get_emb = apply_ma(enc)
print(get_emb.shape)

torch.Size([8, 512])


# **cross-attention**

In [35]:
class CrossAttention:

    def __init__(self, dimen=4, random_w=True):

        self.dimen = dimen
        self.random_w = random_w

        if random_w:
            self.w_query = torch.rand(dimen, dimen).float()
            self.w_key = torch.rand(dimen, dimen).float()
            self.w_value = torch.rand(dimen, dimen).float()
        else:
            self.w_query = None
            self.w_key = None
            self.w_value = None

    def getQKV(self, output_emb, input_emb):

        if self.random_w == False:
            
            queries = output_emb.float()
            keys = input_emb.float()
            values = input_emb.float()
            
        else:
        
            queries = torch.matmul(output_emb.float(), self.w_query)
            keys = torch.matmul(input_emb.float(), self.w_key)
            values = torch.matmul(input_emb.float(), self.w_value)

        return queries, keys, values

    def dotproduct(self, queries, keys):

        dot_product = torch.matmul(queries,keys.T)
        return dot_product

    def get_sqrt(self, dot_product):

        sqroot = math.sqrt(self.dimen)
        return (1/sqroot)*dot_product

    def apply_softmax(self, scaled_dp):
        
        weights = torch.softmax(scaled_dp.T, dim=0).T
        return weights

    def mul_weights_and_values(self, weights, values):

        contextual_embeddings = torch.matmul(weights, values)
        return contextual_embeddings
        

    def __call__(self, output_emb, input_emb):

        queries, keys, values = self.getQKV(output_emb, input_emb)
        dot_product = self.dotproduct(queries,keys)
        scaled_dp = self.get_sqrt(dot_product)
        weights = self.apply_softmax(scaled_dp)
        contextual_embeddings = self.mul_weights_and_values(weights,values)
        
        return contextual_embeddings

In [36]:
sentences = [["How", "are", "you?"], ["I","am","fine."]]
print(sentences)

[['How', 'are', 'you?'], ['I', 'am', 'fine.']]


In [37]:
def get_emb(model, word, pos, dim):
        
    emb = model.wv[word]
    
    pos_enc = np.zeros(dim)
    
    for i in range(dim//2): # 1 pair
        pos_enc[2*i] = math.sin(pos/pow(10000,(2*i/dim)))
        pos_enc[2*i+1] = math.cos(pos/pow(10000,(2*i/dim)))

    emb_np = np.array(emb)
    emb_t = torch.tensor(emb_np)

    pos_np = np.array(pos_enc)
    pos_t = torch.tensor(pos_np)

    result = torch.add(emb_t, pos_t)
    return result


def generate_pos_encodings(model,sentence):

    embeddings = [get_emb(model,word,pos,4) for pos,word in enumerate(sentence)]
    return torch.stack(embeddings)

In [38]:
model = Word2Vec(sentences, vector_size=4, window=5, min_count=1, sg=1)

In [39]:
input_emb = generate_pos_encodings(model,sentences[0])
output_emb = generate_pos_encodings(model,sentences[1])
print(input_emb.shape)
print(output_emb.shape)

torch.Size([3, 4])
torch.Size([3, 4])


In [40]:
ca = CrossAttention(4,True)
get_emb = ca(output_emb,input_emb)
print(get_emb.shape)

torch.Size([3, 4])
